# GraviFrame - DataFrames and time series

*Dibuat oleh Gravicode Studios, dipimpin oleh Kang Fadhil*

In [ ]:
#r "../src/GraviNum/bin/Release/net10.0/Gravicode.Science.GraviNum.dll"
#r "../src/GraviFrame/bin/Release/net10.0/Gravicode.Science.GraviFrame.dll"
#r "nuget: ScottPlot, 5.1.59"

using Gravicode.Science.GraviFrame;
using Gravicode.Science.GraviNum;

var titanic = DataFrame.ReadCsv("../datasets/titanic.csv");
Console.WriteLine(titanic.SelectColumns("survived", "pclass", "sex", "age", "fare").ToString(10));

## Missing values

In [ ]:
foreach (var (column, missing) in titanic.MissingCounts().Where(kv => kv.Value > 0))
    Console.WriteLine($"{column,-16}{missing,5} missing");

var clean = titanic.WithColumn(titanic.Numeric("age").FillMissingWithMedian().Rename("age_filled"));

## GroupBy and pivot

In [ ]:
Console.WriteLine(clean.GroupBy("pclass", "sex").Mean("survived").ToString(10));
Console.WriteLine();
Console.WriteLine(clean.Pivot("pclass", "sex", "survived").ToString());

## Time series

Rolling windows, percentage change and calendar resampling.

In [ ]:
var prices = DataFrame.ReadCsv("../datasets/finance_timeseries.csv");
var grvc = prices.Filter(r => r.String("ticker") == "GRVC").SortBy("date");
var close = grvc.Numeric("close");

var enriched = grvc
    .WithColumn(close.Rolling(7).Mean().Rename("ma7"))
    .WithColumn(close.Rolling(30).Mean().Rename("ma30"))
    .WithColumn(close.PercentChange().Rename("daily_return"));

Console.WriteLine(enriched.SelectColumns("date", "close", "ma7", "ma30").Tail(6).ToString());
Console.WriteLine($"annualised volatility: {enriched.Numeric("daily_return").Std() * Math.Sqrt(252):P2}");

## Trend chart

In [ ]:
var days = Enumerable.Range(0, grvc.RowCount).Select(i => (double)i).ToArray();

var plot = new ScottPlot.Plot();
var c = plot.Add.Scatter(days, close.Values); c.LegendText = "close"; c.MarkerSize = 0;
var m7 = plot.Add.Scatter(days, enriched.Numeric("ma7").Values); m7.LegendText = "7-day"; m7.MarkerSize = 0;
var m30 = plot.Add.Scatter(days, enriched.Numeric("ma30").Values); m30.LegendText = "30-day"; m30.MarkerSize = 0;

plot.Title("GRVC close with rolling averages");
plot.ShowLegend();
plot.GetPngHtml(950, 500)

## Describe and correlate

In [ ]:
Console.WriteLine(clean.SelectColumns("survived", "pclass", "age_filled", "fare").Describe().ToString());
Console.WriteLine();
Console.WriteLine(clean.SelectColumns("survived", "pclass", "age_filled", "fare").CorrelationMatrix().ToString());

## Window functions

Window functions evaluate **within groups**. Two interleaved customers here, so a partition bug
cannot hide: without partitioning, customer `b`'s first row would lag onto customer `a`'s last,
which is the kind of leak that quietly inflates a model's score.


In [ ]:
using Gravicode.Science.GraviFrame;

var sales = new DataFrame(
[
    new TextSeries("customer", ["a", "b", "a", "b", "a", "b"]),
    new NumericSeries("day", [1, 1, 2, 2, 3, 3]),
    new NumericSeries("amount", [10, 100, 20, 200, 30, 300]),
]);

string Show(NumericSeries s) => string.Join(", ",
    s.Values.ToArray().Select(v => double.IsNaN(v) ? "NaN" : v.ToString("G6")));

Console.WriteLine($"CumulativeSum : {Show(Windowing.CumulativeSum(sales, ["customer"], "amount"))}");
Console.WriteLine($"Lag           : {Show(Windowing.Lag(sales, ["customer"], "amount"))}");
Console.WriteLine($"RollingMean(2): {Show(Windowing.RollingMean(sales, ["customer"], "amount", window: 2))}");
Console.WriteLine($"Rank desc     : {Show(Windowing.Rank(sales, ["customer"], "amount", descending: true))}");

Console.WriteLine("\nThe NaNs are each group's first row, and incomplete windows stay missing");
Console.WriteLine("rather than being averaged over whatever happens to be there.");


## As-of join

An equality join on a timestamp matches almost nothing, because two systems never stamp the same
instant. As-of takes the most recent right row at or before each left key.

The direction is **backward-only** on purpose. Matching the *nearest* row in either direction is
look-ahead, and is how a backtest ends up predicting the past.


In [ ]:
var trades = new DataFrame(
[
    new NumericSeries("time", [10, 25, 40]),
    new NumericSeries("size", [1, 2, 3]),
]);

var quotes = new DataFrame(
[
    new NumericSeries("time", [30, 5, 20, 50]),   // deliberately unordered
    new NumericSeries("price", [300, 100, 200, 400]),
]);

Console.WriteLine(Windowing.AsOfJoin(trades, quotes, "time"));
Console.WriteLine("t=40 sees the quote from t=30, never the one from t=50.\n");

var tolerant = Windowing.AsOfJoin(trades, quotes, "time", tolerance: 8);
Console.WriteLine($"with tolerance 8: {Show(tolerant.Numeric("price"))}");
Console.WriteLine("a quote too stale to be useful is worse than no quote, if you say so");


## Categorical columns

Dictionary encoding shrinks a repetitive text column into an `int[]`. The second reason is
**ordering**: a `TextSeries` can only sort alphabetically, which puts "high" before "low" before
"medium" — a real and easily missed wrong answer for ordinal data.


In [ ]:
var sizes = CategoricalSeries.FromValues(
    "size", ["medium", "low", "high", "low", null, "medium"],
    categories: ["low", "medium", "high"], ordered: true);

Console.WriteLine($"categories : [{string.Join(", ", sizes.Categories)}]");
Console.WriteLine($"codes      : [{string.Join(", ", sizes.Codes.ToArray())}]   (-1 is missing)");
Console.WriteLine($"ordered    : [{string.Join(", ", sizes.ArgSort().Select(i => sizes[i] ?? "<missing>"))}]");

Console.WriteLine("\ncounts, including categories with no rows at all:");
foreach (var (category, count) in sizes.CategoryCounts())
    Console.WriteLine($"  {category,-8} {count}");

Console.WriteLine($"\nOneHot(dropFirst: true) -> {string.Join(", ", sizes.OneHot(dropFirst: true).Select(c => c.Name))}");
Console.WriteLine("the dropped category is the baseline; keeping all of them alongside an");
Console.WriteLine("intercept makes the design matrix rank-deficient");


## Excel round trip

`.xlsx` is a zip of XML parts, all of which the BCL already reads — so no spreadsheet library is
needed. The reader handles the three things that catch people out: absent cells are gaps rather than
blanks, dates are numbers marked only by a style, and Excel believes 1900 was a leap year.


In [ ]:
using Gravicode.Science.GraviFrame.Io;

var report = new DataFrame(
[
    new TextSeries("city", ["Bandung", "Jakarta", "Surabaya"]),
    new NumericSeries("population", [2.5e6, 10.6e6, 2.9e6]),
    new DateTimeSeries("surveyed", [new DateTime(2024, 3, 1), new DateTime(2024, 6, 15), null]),
]);

var workbook = Path.Combine(Path.GetTempPath(), $"gravi_{Guid.NewGuid():N}.xlsx");
ExcelWriter.Write(report, workbook, sheetName: "Cities");

Console.WriteLine($"sheets: [{string.Join(", ", ExcelReader.SheetNames(workbook))}]");
var back = ExcelReader.Read(workbook);
Console.WriteLine(back);
Console.WriteLine($"the missing date came back missing: {back["surveyed"].IsMissing(2)}");

File.Delete(workbook);


## Rolling volatility

Window functions on real data. A 21-day rolling standard deviation of daily returns, which is the
usual way volatility is actually measured.


In [ ]:
var priceFrame = DataFrame.ReadCsv("../datasets/finance_timeseries.csv");

var closing = priceFrame.Numeric("close");
var returns = new double[closing.Length];
for (var i = 1; i < closing.Length; i++) returns[i] = (closing[i] - close[i - 1]) / close[i - 1];

var withReturns = new DataFrame([new NumericSeries("ret", returns)]);
var meanReturn = Windowing.RollingMean(withReturns, [], "ret", window: 21);

// Rolling variance from the rolling mean of squares minus the square of the rolling mean.
var squares = new DataFrame([new NumericSeries("sq", returns.Select(r => r * r).ToArray())]);
var meanSquare = Windowing.RollingMean(squares, [], "sq", window: 21);

var dayIndex = new List<double>();
var annualised = new List<double>();
for (var i = 0; i < closing.Length; i++)
{
    if (double.IsNaN(meanSquare[i]) || double.IsNaN(meanReturn[i])) continue;
    var variance = Math.Max(0, meanSquare[i] - meanReturn[i] * meanReturn[i]);
    dayIndex.Add(i);
    annualised.Add(Math.Sqrt(variance * 252));   // annualised
}

var volPlot = new ScottPlot.Plot();
volPlot.Add.Scatter(dayIndex.ToArray(), annualised.ToArray()).MarkerSize = 0;
volPlot.Title("21-day rolling volatility, annualised");
volPlot.XLabel("trading day");
volPlot.YLabel("volatility");
volPlot.GetPngHtml(850, 400)


## Arrow interchange

Arrow is how a dataframe crosses a language boundary without being serialised to text. The check
that matters is not the round trip — a round trip through one implementation agrees with itself by
construction — but whether *pyarrow* reads it. `tools/verify/arrow_interop.py` runs 21 checks in
both directions.

In [ ]:
using Gravicode.Science.GraviFrame.Io;

var arrowFrame = new DataFrame(
[
    new TextSeries("symbol", ["BBCA", "TLKM", "ASII", null]),
    new NumericSeries("close", [9250.0, 3120.0, double.NaN, 4410.0]),
    new BooleanSeries("halted", [false, false, true, null]),
    new DateTimeSeries("stamp",
        [new DateTime(2024, 1, 2), new DateTime(2024, 1, 3), new DateTime(2024, 1, 4), null]),
]);

var arrowPath = Path.Combine(Path.GetTempPath(), "gravi-notebook.arrow");
ArrowFile.Write(arrowFrame, arrowPath);
var arrowBack = ArrowFile.Read(arrowPath);

Console.WriteLine($"{new FileInfo(arrowPath).Length} bytes of Arrow IPC");
Console.WriteLine(arrowBack.ToString());
Console.WriteLine(string.Join(", ", arrowBack.ColumnNames.Select(n => $"{n}={arrowBack[n].DataType}")));
Console.WriteLine($"missing values survived: symbol[3]={arrowBack["symbol"].IsMissing(3)}, " +
                  $"close[2]={arrowBack["close"].IsMissing(2)}, stamp[3]={arrowBack["stamp"].IsMissing(3)}");

File.Delete(arrowPath);

## Out-of-core aggregation

`ChunkedFrame` reads a CSV a block at a time; `Streaming` aggregates over the blocks. Memory is
bounded by the number of distinct groups, not by the number of rows — so check the key's
cardinality first, because that number is the difference between a query that runs and one that
does not.

In [ ]:
var chunkedTitanic = ChunkedFrame.FromCsv("../datasets/titanic.csv", chunkRows: 64);
Console.WriteLine($"{Streaming.CountRows(chunkedTitanic)} rows counted, 64 at a time");

foreach (var (name, stats) in Streaming.Describe(chunkedTitanic, ["fare", "age"]).OrderBy(p => p.Key))
    Console.WriteLine($"  {name,-5} n={stats.Count,4} mean={stats.Mean,8:F3} sd={stats.StandardDeviation,8:F3}");

Console.WriteLine($"distinct (pclass, sex) groups: {Streaming.CountGroups(chunkedTitanic, ["pclass", "sex"])}");

var streamedGroups = Streaming
    .GroupBy(chunkedTitanic, ["pclass", "sex"], ("fare", "mean"), ("survived", "mean"))
    .SortBy([("pclass", true), ("sex", true)]);

Console.WriteLine(streamedGroups.ToString(12));

// Chunk size is a memory knob, not a parameter of the answer. That only holds because
// ChunkedFrame pins its column types from one sample rather than inferring them per chunk.
foreach (var rows in new[] { 8, 512, 4096 })
{
    var probe = ChunkedFrame.FromCsv("../datasets/titanic.csv", chunkRows: rows);
    Console.WriteLine($"  chunkRows={rows,5} -> mean fare {Streaming.Describe(probe, ["fare"])["fare"].Mean:F9}");
}

In [ ]:
var groupIndex = Enumerable.Range(0, streamedGroups.RowCount).Select(i => (double)i).ToArray();
var groupFares = streamedGroups.Numeric("fare_mean");
var groupSurvival = streamedGroups.Numeric("survived_mean");

var streamPlot = new ScottPlot.Plot();
var fareBars = streamPlot.Add.Bars(groupIndex, groupIndex.Select((_, i) => groupFares[i]).ToArray());
fareBars.LegendText = "mean fare";

var survivalMarks = streamPlot.Add.Scatter(groupIndex, groupIndex.Select((_, i) => groupSurvival[i] * 100).ToArray());
survivalMarks.LegendText = "survival rate (%)";
survivalMarks.MarkerSize = 9;

streamPlot.Axes.Bottom.SetTicks(groupIndex,
    Enumerable.Range(0, streamedGroups.RowCount)
        .Select(i => $"{streamedGroups["pclass"].GetValue(i)} {streamedGroups["sex"].GetValue(i)}")
        .ToArray());
streamPlot.Title("Streamed group-by, 64 rows in memory at a time");
streamPlot.ShowLegend();
streamPlot.GetPngHtml(900, 500)